<!-- Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. -->

# MJX 03 — Control & Inverse Kinematics (SO101 + XLeRobot)

Goal: go from *setting joint targets* to *commanding an end-effector pose*.

- **Joint-space control**: write `data.ctrl` (position actuators) to drive joints.
- **Inverse Kinematics (IK)**: given a target end-effector position, solve for joint angles. We use **Damped Least Squares (DLS)** with MuJoCo's analytic Jacobian (`mj_jacBody`) — the standard, robust method that transfers across robots.

Two attempts on two real robots:
1. **SO-101** (SO-ARM100) — a 5-DOF single arm + jaw, loaded from MuJoCo Menagerie via `robot_descriptions`.
2. **XLeRobot** — a dual 5-DOF-arm mobile manipulator, loaded from the XLeRobot repo's MuJoCo model.

Everything renders headless on the GPU via EGL.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np
import mujoco
import imageio

os.makedirs("output/videos", exist_ok=True)


def joint_addrs(model, names):
    """Map joint names to their qpos and dof (Jacobian column) addresses."""
    qadr, dadr = [], []
    for n in names:
        jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, n)
        qadr.append(int(model.jnt_qposadr[jid]))
        dadr.append(int(model.jnt_dofadr[jid]))
    return qadr, dadr


def dls_ik_step(model, data, ee_body_id, target_pos, qadr, dadr, damping=0.15, step=0.5):
    """One Damped Least Squares IK update toward a 3D target position."""
    mujoco.mj_forward(model, data)
    err = target_pos - data.xpos[ee_body_id]
    jacp = np.zeros((3, model.nv))
    jacr = np.zeros((3, model.nv))
    mujoco.mj_jacBody(model, data, jacp, jacr, ee_body_id)
    J = jacp[:, dadr]
    dq = J.T @ np.linalg.solve(J @ J.T + (damping ** 2) * np.eye(3), err)
    for k, a in enumerate(qadr):
        data.qpos[a] += step * dq[k]
    return float(np.linalg.norm(err))


def fit_free_camera(model, data, dist=None, elev=-20.0, azim=120.0, lookat=None):
    """A free camera that auto-frames the model, with optional distance/lookat override."""
    cam = mujoco.MjvCamera()
    mujoco.mjv_defaultFreeCamera(model, cam)
    if dist is not None:
        cam.distance = dist
    if lookat is not None:
        cam.lookat[:] = lookat
    cam.elevation = elev
    cam.azimuth = azim
    return cam


def visual_only_option(model):
    """Hide collision shapes by showing only the model's visual geom group.

    Some models (e.g. SO-ARM100) put visual meshes in a dedicated non-zero
    group; others (e.g. XLeRobot) keep every geom in group 0. When there is no
    separate visual group we just show all groups."""
    import collections
    opt = mujoco.MjvOption()
    counts = collections.Counter(int(model.geom_group[i]) for i in range(model.ngeom))
    nonzero = [g for g in counts if g != 0]
    if not nonzero:
        opt.geomgroup[:] = 1
        return opt
    visual_grp = max(nonzero, key=lambda g: counts[g])
    opt.geomgroup[:] = 0
    opt.geomgroup[visual_grp] = 1
    return opt

## Attempt 1 — SO-101 (SO-ARM100)

A 5-DOF arm (`Rotation`, `Pitch`, `Elbow`, `Wrist_Pitch`, `Wrist_Roll`) plus a `Jaw` gripper. We drive the arm's end-effector along a small square trajectory using DLS IK.

In [ ]:
# robot_descriptions auto-downloads the SO-ARM100 MJCF from MuJoCo Menagerie.
from robot_descriptions import so_arm100_mj_description as so_arm

so_model = mujoco.MjModel.from_xml_path(so_arm.MJCF_PATH)
so_data = mujoco.MjData(so_model)
mujoco.mj_forward(so_model, so_data)

ARM_JOINTS = ["Rotation", "Pitch", "Elbow", "Wrist_Pitch", "Wrist_Roll"]
so_qadr, so_dadr = joint_addrs(so_model, ARM_JOINTS)
# End-effector proxy: the last body in the kinematic tree (around the jaw).
so_ee = so_model.nbody - 1
print("SO-101 end-effector body:", mujoco.mj_id2name(so_model, mujoco.mjtObj.mjOBJ_BODY, so_ee))
print("start EE pos:", np.round(so_data.xpos[so_ee], 3))

In [ ]:
so_renderer = mujoco.Renderer(so_model, height=320, width=320)
so_opt = visual_only_option(so_model)
so_cam = fit_free_camera(so_model, so_data, dist=0.7, elev=-20, azim=-60)

# A small square trajectory in the vertical plane, centered near the start pose.
center = so_data.xpos[so_ee].copy()
r = 0.08
square = []
for t in np.linspace(0, 1, 80):
    ang = 2 * np.pi * t
    square.append(center + np.array([0.0, r * np.cos(ang), r * np.sin(ang)]))

frames, errs = [], []
for target in square:
    for _ in range(6):  # a few IK iterations per waypoint
        e = dls_ik_step(so_model, so_data, so_ee, target, so_qadr, so_dadr, step=0.6)
    errs.append(e)
    so_renderer.update_scene(so_data, camera=so_cam, scene_option=so_opt)
    frames.append(so_renderer.render())

imageio.mimsave("output/videos/mjx03_so101.mp4", frames, fps=20)
print(f"SO-101: {len(frames)} frames, final tracking error = {errs[-1]*1000:.1f} mm")

In [ ]:
from IPython.display import Video
Video(url="output/videos/mjx03_so101.mp4")

## Attempt 2 — XLeRobot (dual-arm mobile manipulator)

XLeRobot has a mobile base (sliders + wheels) and two 5-DOF arms (`*_R` / `*_L`) each with a jaw. We download its MuJoCo model from the upstream repo and run DLS IK on the **right arm** to reach a target, keeping the base and left arm fixed.

The default render shows collision shapes, so we reuse the visual-group trick from MJX 02 and a fitted free camera.

In [ ]:
import subprocess, os

# Sparse-clone only the MuJoCo model directory from the XLeRobot repo.
if not os.path.exists("XLeRobot/simulation/mujoco/scene.xml"):
    subprocess.run(
        "git clone --depth 1 --filter=blob:none --sparse "
        "https://github.com/Vector-Wangel/XLeRobot.git XLeRobot",
        shell=True, check=True,
    )
    subprocess.run(
        "cd XLeRobot && git sparse-checkout set simulation/mujoco",
        shell=True, check=True,
    )

xle_model = mujoco.MjModel.from_xml_path("XLeRobot/simulation/mujoco/scene.xml")
xle_data = mujoco.MjData(xle_model)
mujoco.mj_forward(xle_model, xle_data)
print("XLeRobot: nq", xle_model.nq, "nu", xle_model.nu, "nbody", xle_model.nbody)

In [ ]:
# Right-arm joints (exclude the jaw/gripper from position IK).
RIGHT_ARM = ["Rotation_R", "Pitch_R", "Elbow_R", "Wrist_Pitch_R", "Wrist_Roll_R"]
xle_qadr, xle_dadr = joint_addrs(xle_model, RIGHT_ARM)

# End-effector proxy: the body of the last right-arm joint (wrist roll link).
wrist_jid = mujoco.mj_name2id(xle_model, mujoco.mjtObj.mjOBJ_JOINT, "Wrist_Roll_R")
xle_ee = int(xle_model.jnt_bodyid[wrist_jid])
print("XLeRobot right-arm EE body:", mujoco.mj_id2name(xle_model, mujoco.mjtObj.mjOBJ_BODY, xle_ee))
print("start EE pos:", np.round(xle_data.xpos[xle_ee], 3))

In [ ]:
xle_renderer = mujoco.Renderer(xle_model, height=360, width=480)
xle_opt = visual_only_option(xle_model)
# Focus the camera on the right arm (the part doing IK) so the cart box
# doesn't dominate the frame.
xle_cam = fit_free_camera(xle_model, xle_data, dist=0.6, elev=-15, azim=150,
                          lookat=xle_data.xpos[xle_ee].copy())

# Reach toward a target offset from the right-arm start pose.
start = xle_data.xpos[xle_ee].copy()
goal = start + np.array([0.10, -0.05, 0.12])

frames, errs = [], []
for t in np.linspace(0, 1, 90):
    target = start + t * (goal - start)
    e = dls_ik_step(xle_model, xle_data, xle_ee, target, xle_qadr, xle_dadr, step=0.4)
    errs.append(e)
    xle_renderer.update_scene(xle_data, camera=xle_cam, scene_option=xle_opt)
    frames.append(xle_renderer.render())

imageio.mimsave("output/videos/mjx03_xlerobot.mp4", frames, fps=20)
print(f"XLeRobot: {len(frames)} frames, final tracking error = {errs[-1]*1000:.1f} mm")

In [ ]:
from IPython.display import Video
Video(url="output/videos/mjx03_xlerobot.mp4")